In [0]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, ArrayType

# 1. 生成模拟资产数据
np.random.seed(42)
n_days = 252
asset_names = ['Asset_A', 'Asset_B', 'Asset_C', 'Asset_D']
num_assets = len(asset_names)

true_means = [0.0005, 0.0008, 0.0003, 0.0006]
true_cov = [
    [0.00040, 0.00010, 0.00005, 0.00010],
    [0.00010, 0.00090, 0.00020, 0.00030],
    [0.00005, 0.00020, 0.00025, 0.00010],
    [0.00010, 0.00030, 0.00010, 0.00050]
]

simulated_returns = np.random.multivariate_normal(true_means, true_cov, size=n_days)
pdf = pd.DataFrame(simulated_returns, columns=asset_names)
df_spark = spark.createDataFrame(pdf)

# 2. 计算协方差与均值
psdf = df_spark.pandas_api()
cov_matrix = psdf[asset_names].cov().to_numpy()
mean_returns = psdf[asset_names].mean().to_numpy()

# 3. SLSQP 二次规划求解最小方差
def portfolio_variance(weights, cov):
    return np.dot(weights.T, np.dot(cov, weights))

init_weights = np.array([1.0 / num_assets] * num_assets)
bounds = tuple((0.0, 1.0) for _ in range(num_assets))
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})

opt_result = minimize(portfolio_variance, init_weights, args=(cov_matrix,), method='SLSQP', bounds=bounds, constraints=constraints)
opt_weights = opt_result.x
min_daily_variance = opt_result.fun

print("================== 优化结果：最小方差组合 ==================")
for name, w in zip(asset_names, opt_weights):
    print(f"资产 {name} 最优权重: {w:.4f} ({w*100:.2f}%)")
print(f"最小日收益率方差: {min_daily_variance:.8f}")

# 4. Serverless 兼容的蒙特卡洛模拟（基于 DataFrame & UDF）
num_simulations = 100000

# 定义 UDF 返回数据结构
schema = StructType([
    StructField("Expected_Return", DoubleType(), False),
    StructField("Variance", DoubleType(), False),
    StructField("Weights", ArrayType(DoubleType()), False)
])

@F.udf(returnType=schema)
def simulate_portfolio_udf(seed):
    np.random.seed(seed)
    w = np.random.random(num_assets)
    w = w / np.sum(w)
    var = float(np.dot(w.T, np.dot(cov_matrix, w)))
    ret = float(np.dot(w, mean_returns))
    return (ret, var, [float(x) for x in w])

# 使用 spark.range 替代 sparkContext.parallelize
sim_df = spark.range(0, num_simulations, numPartitions=16) \
              .withColumn("result", simulate_portfolio_udf(F.col("id"))) \
              .select("result.*")

best_sim = sim_df.orderBy("Variance").first()

print("\n================== 蒙特卡洛模拟验证 (Serverless) ==================")
print(f"模拟找到的最小日方差: {best_sim['Variance']:.8f}")
print("对应的近似权重分布:", [round(x, 4) for x in best_sim['Weights']])

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: `to_numpy` loads all data into the driver's memory. It should only be used if the resulting NumPy ndarray is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/databricks/python/lib/python3.12/site-packages/pyspark/pandas/utils.py:1055: PandasAPIOnSparkAdviceWarning: `to_numpy` loads all data into the driver's memory. It should only be used if the resulting NumPy ndarray is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


================== 优化结果：最小方差组合 ==================
资产 Asset_A 最优权重: 0.2500 (25.00%)
资产 Asset_B 最优权重: 0.2500 (25.00%)
资产 Asset_C 最优权重: 0.2500 (25.00%)
资产 Asset_D 最优权重: 0.2500 (25.00%)
最小日收益率方差: 0.00020595

================== 蒙特卡洛模拟验证 (Serverless) ==================
模拟找到的最小日方差: 0.00015421
对应的近似权重分布: [0.2586, 0.0036, 0.5177, 0.22]
